In [11]:
from selenium import webdriver

In [12]:
browser = webdriver.Chrome()

In [13]:
browser.get('https://www.naver.com/')

In [14]:
# 1. 네이버에서 검색어 입력칸 선택
# 2. 선택한 검색어 입력칸에 검색어 입력하기

In [15]:
# 1. 네이버에서 검색어 입력칸 선택
input_word_place = browser.find_elements('css selector', '#query')[0] # #: id 속성
input_word_place

<selenium.webdriver.remote.webelement.WebElement (session="15ab1899b766b0db1ce97e7919c6ebd8", element="f.2E1030C7D0A9396206DF98C992C7BFB1.d.77E2AC48C17F444FD55CF1A20FE8B2DD.e.102")>

In [16]:
# 2. 선택한 검색어 입력칸에 검색어 입력하기
searching = '파이썬'
input_word_place.send_keys(searching)

In [17]:
searching = '주식'
input_word_place.clear()
input_word_place.send_keys(searching)

In [18]:
searching = '부동산'
input_word_place = browser.find_elements('css selector', '#query')[0]
input_word_place.clear()
input_word_place.send_keys(searching)

In [19]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(browser.page_source, 'html.parser')

# 연관검색어 부분/파트 찾기
tag_list = soup.select('li.item._item')
#len(tag_list)
tag = tag_list[1]
related_word = tag.select('span.kwd_txt')[0].text
related_word

'주식시세'

In [20]:
soup = BeautifulSoup(browser.page_source, 'html.parser')

# 연관검색어 부분/파트 찾기
tag_list = soup.select('li.item._item')
for tag in tag_list:
    related_word = tag.select('span.kwd_txt')[0].text
    print(related_word)

부동산
부동산무료법률상담
부동산상가
부동산홈
부동산실거래가조회
지방선거부동산
금리인상부동산


In [21]:
# 코드 정리(1차)
from selenium import webdriver
from bs4 import BeautifulSoup
import time

browser = webdriver.Chrome()
browser.get('https://www.naver.com/')
#0.5초만 쉬어줘
time.sleep(0.5)

# 검색어입력
searching = '주식'
input_word_place = browser.find_elements('css selector', '#query')[0]
input_word_place.clear()
input_word_place.send_keys(searching)
time.sleep(1)       # 1초 쉬어줘   <---위치 중요

# 연관검색어 찾아서 출력하기
soup = BeautifulSoup(browser.page_source, 'html.parser')
tag_list = soup.select('li.item._item')
results = []
for tag in tag_list:
    related_word = tag.select('span.kwd_txt')[0].text
    data = [searching, related_word]
    # searching과 related_wordrk 다른 경우에만 results에 추가하기
    if searching != related_word:
        results.append(data)

results

[['주식', '주식시세'],
 ['주식', '주식사는법'],
 ['주식', '주식분석'],
 ['주식', '국내주식'],
 ['주식', '주식시장'],
 ['주식', '철도주식']]

## 함수 만들기

In [22]:
def get_related_word(browser, searching):
    # 검색어 입력
    input_word_place = browser.find_elements('css selector', '#query')[0]  
    input_word_place.clear()
    input_word_place.send_keys(searching)
    time.sleep(1)
    
    soup = BeautifulSoup(browser.page_source, 'html.parser')
    tag_list = soup.select('li.item._item')
    results = []
    for tag in tag_list:
        related_word = tag.select('span.kwd_txt')[0].text
        data = [searching, related_word]
        # searching과 related_wordrk 다른 경우에만 results에 추가하기
        if searching != related_word:
            results.append(data)

        return results

In [23]:
browser = webdriver.Chrome()
browser.get('https://www.naver.com/')
time.sleep(0.5)

searching = '파이썬'
# 1차 연관검색어 수집
related_word_list = get_related_word(browser, searching)

In [24]:
results = []

# 1차 연관검색어 추가하기
results = results + related_word_list
for related_word_part in related_word_list:
    related_word = related_word_part[1]
   # print(related_word)

    related_word_list2 = get_related_word(browser, related_word)
   # print(related_word_list2)
    results = results + related_word_list2

In [25]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

In [26]:
browser = webdriver.Chrome()
browser.get('https:/naver.com')
time.sleep(0.5)

In [27]:
def get_related_word(browser, searching):
    
    # 검색어 입력
    input_word_place = browser.find_elements('css selector', '#query')[0]  
    input_word_place.clear( )
    input_word_place.send_keys(searching)
    time.sleep(1)   # 1초 쉬어줘!!!  

    # 연관검색어 찾아서 출력하기
    soup = BeautifulSoup( browser.page_source, "html.parser")
    tag_list = soup.select('li.item._item')
    results =  [ ]
    for tag in tag_list:
        related_word = tag.select('span.kwd_txt')[0].text
        data = [searching, related_word ]
        if searching != related_word:
            results.append(data)

    return results

In [28]:
# 브라우저 열기
browser = webdriver.Chrome()
browser.get('https:/naver.com')
time.sleep(0.5) # 0.5초만 쉬어줘!!!

searching = '파이썬'

# 1차 연관검색어 수집
related_word_list = get_related_word(browser, searching)
results =  [ ]
results = results + related_word_list
# 2차연관검색어 수집하기
for related_word_part in related_word_list:
    related_word = related_word_part[1]
    related_word_list2 = get_related_word(browser, related_word)
    results = results + related_word_list2
    


In [29]:
# 표형태로 저장 -> 엑셀 파일에 저장하기
import pandas as pd

df = pd.DataFrame(results)
df.columns = ['검색어', '연관검색어']
df.to_excel('./related_word_[searching].xlsx', index = False)

In [30]:
df

,검색어,연관검색어
0,파이썬,파이썬 뜻
1,파이썬,파이썬 독학
2,파이썬,파이썬 자격증
3,파이썬,파이썬 기초
4,파이썬,파이썬 다운로드
...,...,...
68,파이썬 설치,파이썬 설치 오류
69,파이썬 설치,파이썬 설치하는 방법
70,파이썬 설치,파이썬 설치 안됨
71,파이썬 설치,파이썬 설치 경로
